In [ ]:
!pip install mysql-connector-python



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.9/33.9 MB 14.2 MB/s eta 0:00:00


In [ ]:
import sqlite3
import pandas as pd


In [ ]:
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON;")

In [ ]:
cursor.execute("""
CREATE TABLE Address (
    address_id INTEGER PRIMARY KEY AUTOINCREMENT,
    street TEXT,
    city TEXT,
    country TEXT,
    postcode TEXT
);
""")

cursor.execute("""
CREATE TABLE Calendar (
    datetime_key TEXT PRIMARY KEY,
    day_name TEXT,
    month INTEGER,
    quarter INTEGER,
    year INTEGER
);
""")
cursor.execute("""
CREATE TABLE Patient (
    patient_id INTEGER PRIMARY KEY AUTOINCREMENT,
    full_name TEXT NOT NULL,
    date_of_birth TEXT NOT NULL,
    gender TEXT CHECK(gender IN ('M','F','O')),
    address_id INTEGER,
    disease_name TEXT,
    disease_description TEXT,
    FOREIGN KEY (address_id) REFERENCES Address(address_id)
);
""")

cursor.execute("""
CREATE TABLE Doctor (
    doctor_id INTEGER PRIMARY KEY AUTOINCREMENT,
    full_name TEXT,
    specialty TEXT,
    phone_no TEXT,
    address_id INTEGER,
    FOREIGN KEY (address_id) REFERENCES Address(address_id)
);
""")

cursor.execute("""
CREATE TABLE SurgicalStaff (
    staff_id INTEGER PRIMARY KEY AUTOINCREMENT,
    full_name TEXT,
    role TEXT
);
""")

cursor.execute("""
CREATE TABLE InsurancePolicy (
    policy_id INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id INTEGER,
    policy_number TEXT,
    provider TEXT,
    FOREIGN KEY (patient_id) REFERENCES Patient(patient_id)
);
""")

cursor.execute("""
CREATE TABLE Treatment (
    treatment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    visit_id INTEGER,
    treatment_type TEXT CHECK(treatment_type IN ('medication','surgery')),
    treatment_cost REAL,
    FOREIGN KEY (visit_id) REFERENCES Visit(visit_id)
);
""")

cursor.execute("""
CREATE TABLE Medication (
    treatment_id INTEGER PRIMARY KEY,
    drug_name TEXT,
    dosage INTEGER,
    frequency TEXT,
    method_of_administration TEXT,
    duration_days INTEGER,
    FOREIGN KEY (treatment_id) REFERENCES Treatment(treatment_id)
);
""")

cursor.execute("""
CREATE TABLE Surgery (
    treatment_id INTEGER PRIMARY KEY,
    surgery_type TEXT,
    surgery_date TEXT,
    anesthesia_used TEXT,
    duration_minutes INTEGER,
    FOREIGN KEY (treatment_id) REFERENCES Treatment(treatment_id)
);
""")

cursor.execute("""
CREATE TABLE Visit (
    visit_id INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id INTEGER,
    doctor_id INTEGER,
    datetime_key TEXT,
    reason TEXT,
    duration_minutes INTEGER,
    follow_up TEXT,
    FOREIGN KEY (patient_id) REFERENCES Patient(patient_id),
    FOREIGN KEY (doctor_id) REFERENCES Doctor(doctor_id),
    FOREIGN KEY (datetime_key) REFERENCES Calendar(datetime_key)
);
""")

cursor.execute("""
CREATE TABLE SurgeryStaff (
    treatment_id INTEGER,
    staff_id INTEGER,
    PRIMARY KEY (treatment_id, staff_id),
    FOREIGN KEY (treatment_id) REFERENCES Surgery(treatment_id),
    FOREIGN KEY (staff_id) REFERENCES SurgicalStaff(staff_id)
);
""")

conn.commit()
print("✅ All tables created.")


✅ All tables created.


In [ ]:
cursor.executemany("""
INSERT INTO Address (street, city, country, postcode) VALUES (?, ?, ?, ?);
""", [
    ('12 Baker St','London','UK','NW1 5LD'),
    ('55 King Rd','Manchester','UK','M1 4AB'),
    ('101 Main Ave','Birmingham','UK','B4 7PT')
])

cursor.executemany("""
INSERT INTO Calendar (datetime_key, day_name, month, quarter, year) VALUES (?, ?, ?, ?, ?);
""", [
    ('2025-07-01 09:00:00','Tuesday', 7,3,2025),
    ('2025-07-01 14:30:00','Tuesday', 7,3,2025),
    ('2025-07-02 10:15:00','Wednesday',7,3,2025)
])


In [ ]:
cursor.executemany("""
INSERT INTO Patient (full_name, date_of_birth, gender, address_id, disease_name, disease_description)
VALUES (?, ?, ?, ?, ?, ?);
""", [
    ('Maria Rossi','1985-04-11','F',1,'Hypertension','High blood pressure'),
    ('John Smith','1972-09-03','M',2,'T2 Diabetes','Type-2 diabetes'),
    ('Lina Ahmed','1990-01-20','F',None,None,None)
])

cursor.executemany("""
INSERT INTO Doctor (full_name, specialty, phone_no, address_id)
VALUES (?, ?, ?, ?);
""", [
    ('Dr Amir Khan','Cardiology','020-5556-8899',1),
    ('Dr Susan Lee','Endocrinology','0161-777-4455',2),
    ('Dr Peter Ng','General Practice','0121-222-3344',3)
])

cursor.executemany("""
INSERT INTO SurgicalStaff (full_name, role)
VALUES (?, ?);
""", [
    ('Nurse Fiona West','Scrub Nurse'),
    ('Dr Ahmed Aziz','Anesthesiologist'),
    ('Dr Chloe Tan','Surgical Assistant')
])


In [ ]:
cursor.executemany("""
INSERT INTO InsurancePolicy (patient_id, policy_number, provider)
VALUES (?, ?, ?);
""", [
    (1, 'POL12345', 'NHS'),
    (1, 'POL67890', 'Aviva'),
    (2, 'POL54321', 'Bupa')
])


In [ ]:
cursor.executemany("""
INSERT INTO Visit (patient_id, doctor_id, datetime_key, reason, duration_minutes, follow_up)
VALUES (?, ?, ?, ?, ?, ?);
""", [
    (1, 1, '2025-07-01 09:00:00', 'Routine check-up', 30, 'Blood test'),
    (2, 2, '2025-07-01 14:30:00', 'Diabetes consultation', 45, 'Dietary review'),
    (1, 3, '2025-07-02 10:15:00', 'General fatigue', 20, 'Further tests')
])


In [ ]:
cursor.executemany("""
INSERT INTO Treatment (visit_id, treatment_type, treatment_cost)
VALUES (?, ?, ?);
""", [
    (1, 'medication', 300),
    (2, 'surgery', 2500),
    (3, 'medication', 400)
])


In [ ]:
cursor.executemany("""
INSERT INTO Medication (treatment_id, drug_name, dosage, frequency, method_of_administration, duration_days)
VALUES (?, ?, ?, ?, ?, ?);
""", [
    (1, 'Lisinopril', 10, 'once daily', 'oral', 30),
    (3, 'Metformin', 500, 'twice daily', 'oral', 60)
])

cursor.execute("""
INSERT INTO Surgery (treatment_id, surgery_type, surgery_date, anesthesia_used, duration_minutes)
VALUES (?, ?, ?, ?, ?);
""", (2, 'Appendectomy', '2025-07-03', 'General', 90))


In [ ]:
cursor.executemany("""
INSERT INTO SurgeryStaff (treatment_id, staff_id)
VALUES (?, ?);
""", [
    (2, 1),  # Nurse Fiona West
    (2, 2),  # Dr Ahmed Aziz
    (2, 3)   # Dr Chloe Tan
])


In [ ]:
pd.read_sql_query("SELECT * FROM Treatment", conn)


,treatment_id,visit_id,treatment_type,treatment_cost
0,1,1,medication,300.0
1,2,2,surgery,2500.0
2,3,3,medication,400.0


from matplotlib import pyplot as plt
_df_0['treatment_id'].plot(kind='hist', bins=20, title='treatment_id')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_1['visit_id'].plot(kind='hist', bins=20, title='visit_id')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_2['treatment_cost'].plot(kind='hist', bins=20, title='treatment_cost')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
_df_3.groupby('treatment_type').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

...

from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(_df_13['treatment_type'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(_df_13, x='treatment_id', y='treatment_type', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(_df_14['treatment_type'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(_df_14, x='visit_id', y='treatment_type', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(_df_15['treatment_type'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(_df_15, x='treatment_cost', y='treatment_type', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

In [ ]:
tables = cursor.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()
print("📋 Tables in the database:")
for t in tables:
    print("-", t[0])


📋 Tables in the database:
- Address
- sqlite_sequence
- Calendar
- Patient
- Doctor
- SurgicalStaff
- InsurancePolicy
- Treatment
- Medication
- Surgery
- Visit
- SurgeryStaff


In [ ]:
for t in cursor.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall():
    table_name = t[0]
    print(f"\n📊 Contents of '{table_name}':")
    display(pd.read_sql_query(f"SELECT * FROM {table_name};", conn))



📊 Contents of 'Address':


,address_id,street,city,country,postcode
0,1,12 Baker St,London,UK,NW1 5LD
1,2,55 King Rd,Manchester,UK,M1 4AB
2,3,101 Main Ave,Birmingham,UK,B4 7PT



📊 Contents of 'sqlite_sequence':


,name,seq
0,Address,3
1,Patient,3
2,Doctor,3
3,SurgicalStaff,3
4,InsurancePolicy,3
5,Visit,3
6,Treatment,3



📊 Contents of 'Calendar':


...

,treatment_id,surgery_type,surgery_date,anesthesia_used,duration_minutes
0,2,Appendectomy,2025-07-03,General,90



📊 Contents of 'Visit':


,visit_id,patient_id,doctor_id,datetime_key,reason,duration_minutes,follow_up
0,1,1,1,2025-07-01 09:00:00,Routine check-up,30,Blood test
1,2,2,2,2025-07-01 14:30:00,Diabetes consultation,45,Dietary review
2,3,1,3,2025-07-02 10:15:00,General fatigue,20,Further tests



📊 Contents of 'SurgeryStaff':


,treatment_id,staff_id
0,2,1
1,2,2
2,2,3
